In [2]:
!pip install transformers datasets peft accelerate scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 74.4 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    BertForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
import torch
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 데이터셋 로드
dataset = load_dataset('smilegate-ai/kor_unsmile')

# 라벨 정의
unsmile_labels = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
num_labels = len(unsmile_labels)

print(f"Train: {len(dataset['train'])} / Valid: {len(dataset['valid'])}")

Using device: cuda
Train: 15005 / Valid: 3737


In [6]:
model_name = 'smilegate-ai/kor_unsmile'

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 베이스 모델 로드 (multi-label classification)
base_model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)
base_model.config.id2label = {i: label for i, label in enumerate(unsmile_labels)}
base_model.config.label2id = {label: i for i, label in enumerate(unsmile_labels)}

tokenizer_config.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [8]:
# LoRA 설정
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,          # 시퀀스 분류 태스크
    r=16,                                 # LoRA rank (16 권장)
    lora_alpha=32,                        # Alpha 스케일링 (보통 r의 2배)
    lora_dropout=0.1,                     # 드롭아웃
    target_modules=["query", "value"],    # BERT attention에 LoRA 적용
    bias="none",
)

# LoRA 모델 생성
model_lora = get_peft_model(base_model, lora_config)
model_lora.print_trainable_parameters()  # 학습 가능 파라미터 확인
# -> 출력 예: trainable params: 294,912 || all params: 108,604,426 || trainable%: 0.27%

trainable params: 597,514 || all params: 109,523,732 || trainable%: 0.5456


In [10]:
def preprocess_function(examples):
    """배치 단위 전처리"""
    # 텍스트 토큰화
    tokenized = tokenizer(
        examples["문장"],
        truncation=True,
        padding=False,
        max_length=128
    )
    
    # labels를 float 텐서로 변환 (multi-label 학습용)
    tokenized["labels"] = [list(map(float, label)) for label in examples["labels"]]
    
    return tokenized

# 데이터셋 전처리
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# 데이터 콜레이터 (패딩)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/15005 [00:00<?, ? examples/s]

Map:   0%|          | 0/3737 [00:00<?, ? examples/s]

In [12]:
def compute_metrics(eval_pred):
    """Multi-label 평가 메트릭 계산"""
    predictions, labels = eval_pred
    
    # Sigmoid 적용 후 0.5 임계값으로 이진화
    preds = (torch.sigmoid(torch.tensor(predictions)) > 0.5).numpy().astype(int)
    labels = labels.astype(int)
    
    # 전체 weighted 메트릭
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    
    # 🎯 악플/욕설 (index 8) 개별 메트릭 - 핵심!
    abuse_precision, abuse_recall, abuse_f1, _ = precision_recall_fscore_support(
        labels[:, 8], preds[:, 8], average='binary', zero_division=0
    )
    
    # LRAP (Label Ranking Average Precision)
    from sklearn.metrics import label_ranking_average_precision_score
    lrap = label_ranking_average_precision_score(labels, predictions)
    
    return {
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'lrap': lrap,
        'abuse_f1': abuse_f1,
        'abuse_recall': abuse_recall,      # 🎯 핵심 지표!
        'abuse_precision': abuse_precision,
    }

In [14]:
# 학습 설정
training_args = TrainingArguments(
    output_dir="./lora_finetuned",
    num_train_epochs=5,                        # LoRA는 더 많은 epoch 가능
    learning_rate=1e-4,                        # Full FT보다 큰 lr 사용
    per_device_train_batch_size=32,            # L40S는 넉넉하게 사용
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="abuse_recall",      # 🎯 악플 recall 기준으로 최적 모델 선택
    greater_is_better=True,
    logging_steps=50,
    fp16=True,                                 # 혼합 정밀도 학습 (속도 향상)
    report_to="none",                          # wandb 등 리포팅 비활성화
)

# Trainer 생성
trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["valid"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 🚀 학습 시작
trainer.train()

/home/j-i14d105/.local/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_3568242/1800451068.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/j-i14d105/.local/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py", line 96, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/j-i14d105/.local/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/j-i14d105/.local/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1747, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/j-i14d105/.local/lib/python3.12/site-packages/peft/peft_model.py", line 1722, in forward
    return self.base_model(
           ^^^^^^^^^^^^^^^^
  File "/home/j-i14d105/.local/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self

In [ ]:
# 최종 평가
eval_results = trainer.evaluate()
print("\n" + "="*50)
print("📊 LoRA Fine-tuning 결과")
print("="*50)
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")
print("="*50)

In [ ]:
# LoRA 어댑터 저장
trainer.save_model("./lora_finetuned/best")
tokenizer.save_pretrained("./lora_finetuned/best")

print("✅ LoRA 모델 저장 완료!")

In [ ]:
# 병합이 필요한 경우 (배포용)
from peft import PeftModel

# 새로 베이스 모델 로드
base_model_for_merge = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

# LoRA 어댑터 병합
merged_model = PeftModel.from_pretrained(base_model_for_merge, "./lora_finetuned/best")
final_model = merged_model.merge_and_unload()  # 일반 BERT 모델로 변환

# 병합된 모델 저장
final_model.save_pretrained("./lora_merged")
tokenizer.save_pretrained("./lora_merged")

print("✅ 병합된 모델 저장 완료!")

In [ ]:
from transformers import TextClassificationPipeline

# 파이프라인 생성
pipe = TextClassificationPipeline(
    model=model_lora,
    tokenizer=tokenizer,
    device=0,
    return_all_scores=True,
    function_to_apply='sigmoid'
)

# 테스트
test_texts = [
    "야 진짜 빡치네",
    "잘한다 잘해",
    "피해 피해",
    "죽어 죽어",
    "이래서 여자는 게임을 하면 안된다",
]

print("\n📝 추론 테스트")
print("="*60)
for text in test_texts:
    print(f"\n🔹 \"{text}\"")
    results = pipe(text)[0]
    for r in sorted(results, key=lambda x: x['score'], reverse=True)[:3]:
        print(f"   - {r['label']}: {r['score']:.3f}")

In [ ]:
🎯 요약
항목	설명
라이브러리	peft (LoRA 구현)
LoRA Rank	16 (경량화와 성능 균형)
학습률	1e-4 (Full FT보다 크게)
Epoch	5 (LoRA는 과적합에 강함)
핵심 지표	abuse_recall (악플/욕설 탐지율)
예상 시간	~10-15분 (L40S 기준)